In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipe, X, y, cv=cv, scoring='balanced_accuracy')

# Load data
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)  # adjust as needed

X = df.iloc[:, :12].values   # the 12 wavelet coefficient columns
y = df.iloc[:, 12].values    # the label column

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Pipeline: scale + classify
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced'))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Cross-validation for a more robust estimate given small N
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv)
print(f"CV Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

              precision    recall  f1-score   support

         1.0       0.73      0.42      0.54        26
         2.0       0.32      0.64      0.42        11

    accuracy                           0.49        37
   macro avg       0.53      0.53      0.48        37
weighted avg       0.61      0.49      0.50        37

[[11 15]
 [ 4  7]]
CV Accuracy: 0.528 ± 0.065


In [4]:
"""
EEG Planning vs Relax classifier
Handles class imbalance (130 relaxed / 52 motor), compares SVM / RF / LDA,
tunes hyperparameters via GridSearchCV, and evaluates with imbalance-aware metrics.
"""

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

RANDOM_STATE = 42

# -----------------------------
# 1. Load data
# -----------------------------
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)

X = df.iloc[:, :12].values   # 12 wavelet coefficient columns
y = df.iloc[:, 12].values    # label column (1.0 = relaxed, 2.0 = motor)

print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))

# -----------------------------
# 2. Train/test split (stratified, so ratio is preserved)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# -----------------------------
# 3. Define candidate pipelines + grids
# -----------------------------
candidates = {
    'SVM': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(class_weight='balanced', random_state=RANDOM_STATE))
        ]),
        'param_grid': {
            'clf__C': [0.1, 1, 10, 100],
            'clf__gamma': ['scale', 0.001, 0.01, 0.1, 1],
            'clf__kernel': ['rbf', 'linear']
        }
    },
    'RandomForest': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE))
        ]),
        'param_grid': {
            'clf__n_estimators': [100, 300],
            'clf__max_depth': [3, 5, 8, None],
            'clf__min_samples_leaf': [1, 3, 5]
        }
    },
    'LDA': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LinearDiscriminantAnalysis())
        ]),
        'param_grid': {
            'clf__solver': ['svd', 'lsqr']
        }
    }
}

# -----------------------------
# 4. Grid search each candidate, scoring on f1_macro (imbalance-aware)
# -----------------------------
results = {}

for name, cfg in candidates.items():
    print(f"\n{'='*50}\nTuning {name}\n{'='*50}")

    grid = GridSearchCV(
        cfg['pipeline'], cfg['param_grid'],
        cv=cv, scoring='f1_macro', n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    print("Best params:", grid.best_params_)
    print("Best CV f1_macro:", round(grid.best_score_, 3))
    print("\nTest set report:")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

    bal_acc = balanced_accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')

    results[name] = {
        'model': best_model,
        'best_params': grid.best_params_,
        'cv_f1_macro': grid.best_score_,
        'test_balanced_accuracy': bal_acc,
        'test_f1_macro': f1_macro
    }

# -----------------------------
# 5. Compare all models side by side
# -----------------------------
print(f"\n{'='*50}\nSUMMARY\n{'='*50}")
summary = pd.DataFrame({
    name: {
        'CV f1_macro': res['cv_f1_macro'],
        'Test balanced_accuracy': res['test_balanced_accuracy'],
        'Test f1_macro': res['test_f1_macro']
    }
    for name, res in results.items()
}).T.sort_values('Test f1_macro', ascending=False)

print(summary.round(3))

best_name = summary.index[0]
print(f"\nBest model: {best_name}")
print(f"Best params: {results[best_name]['best_params']}")

# -----------------------------
# 6. Final cross-validated metrics for the winning model (accuracy / balanced_accuracy / f1_macro)
# -----------------------------
best_pipeline = results[best_name]['model']
for scoring in ['accuracy', 'balanced_accuracy', 'f1_macro']:
    scores = cross_val_score(best_pipeline, X, y, cv=cv, scoring=scoring)
    print(f"{scoring}: {scores.mean():.3f} ± {scores.std():.3f}")

Class distribution: {np.float64(1.0): np.int64(130), np.float64(2.0): np.int64(52)}

Tuning SVM
Best params: {'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'linear'}
Best CV f1_macro: 0.459

Test set report:
              precision    recall  f1-score   support

         1.0       0.71      0.38      0.50        26
         2.0       0.30      0.64      0.41        11

    accuracy                           0.46        37
   macro avg       0.51      0.51      0.46        37
weighted avg       0.59      0.46      0.47        37

Confusion matrix:
 [[10 16]
 [ 4  7]]

Tuning RandomForest
Best params: {'clf__max_depth': 5, 'clf__min_samples_leaf': 3, 'clf__n_estimators': 300}
Best CV f1_macro: 0.47

Test set report:
              precision    recall  f1-score   support

         1.0       0.69      0.69      0.69        26
         2.0       0.27      0.27      0.27        11

    accuracy                           0.57        37
   macro avg       0.48      0.48      0.48        3

In [7]:
"""
Replicating Bayram et al. approach on the Planning Relax dataset:
1. Sequential Feature Selection (SFS)
2. t-test / p-value based feature ranking
3. SVM kernel comparison (linear, RBF, poly)
"""

import pandas as pd
import numpy as np
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

RANDOM_STATE = 42

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv('plrx.txt', delimiter='\t', header=None).iloc[:, :13]
X = df.iloc[:, :12].values
y = df.iloc[:, 12].values
feature_names = [f'coef_{i+1}' for i in range(12)]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)


# =====================================================================
# STEP 1: Sequential Feature Selection (SFS)
# =====================================================================
# Forward SFS: starts with 0 features, greedily adds the feature that
# most improves CV score at each step, until k_features is reached.
print("="*60)
print("STEP 1: Sequential Feature Selection")
print("="*60)

base_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', class_weight='balanced', random_state=RANDOM_STATE))
])

# Try SFS for a range of feature-set sizes and see which gives best CV f1_macro
best_k, best_score, best_support = None, -np.inf, None

for k in range(2, 12):  # try keeping 2 to 12 features
    sfs = SequentialFeatureSelector(
        base_svm, n_features_to_select=k, direction='forward',
        scoring='f1_macro', cv=cv, n_jobs=-1
    )
    sfs.fit(X_train, y_train)
    selected = sfs.get_support()

    # Evaluate this feature subset with CV
    scores = cross_val_score(base_svm, X_train[:, selected], y_train, cv=cv, scoring='f1_macro')
    mean_score = scores.mean()

    print(f"k={k:2d} | features={[feature_names[i] for i in range(12) if selected[i]]} | "
          f"CV f1_macro={mean_score:.3f}")

    if mean_score > best_score:
        best_score, best_k, best_support = mean_score, k, selected

print(f"\nBest SFS result: k={best_k}, CV f1_macro={best_score:.3f}")
print("Selected features:", [feature_names[i] for i in range(12) if best_support[i]])

X_train_sfs = X_train[:, best_support]
X_test_sfs = X_test[:, best_support]


# =====================================================================
# STEP 2: t-test / p-value based feature ranking
# =====================================================================
# For each feature, run an independent t-test between class 1 and class 2.
# Features with low p-values (< 0.05) differ significantly between classes
# and are more likely to be useful for classification.
print("\n" + "="*60)
print("STEP 2: t-test / p-value feature ranking")
print("="*60)

class1 = X_train[y_train == 1.0]
class2 = X_train[y_train == 2.0]

pvalues = []
tstats = []
for i in range(X_train.shape[1]):
    t_stat, p_val = stats.ttest_ind(class1[:, i], class2[:, i], equal_var=False)
    tstats.append(t_stat)
    pvalues.append(p_val)

ttest_results = pd.DataFrame({
    'feature': feature_names,
    't_stat': tstats,
    'p_value': pvalues
}).sort_values('p_value')

print(ttest_results.to_string(index=False))

# Keep only statistically significant features (p < 0.05)
significant_features = ttest_results[ttest_results['p_value'] < 0.05]['feature'].tolist()
significant_idx = [feature_names.index(f) for f in significant_features]

print(f"\nSignificant features (p < 0.05): {significant_features}")

if len(significant_idx) == 0:
    print("No features passed p < 0.05 — falling back to top 4 by p-value for comparison.")
    significant_idx = ttest_results.index[:4].tolist()

X_train_ttest = X_train[:, significant_idx]
X_test_ttest = X_test[:, significant_idx]


# =====================================================================
# STEP 3: SVM kernel comparison
# =====================================================================
# Compare linear / rbf / poly kernels, on: (a) all features, (b) SFS-selected,
# (c) t-test-selected, to see which feature set + kernel combo works best.
print("\n" + "="*60)
print("STEP 3: SVM kernel comparison")
print("="*60)

feature_sets = {
    'All 12 features': (X_train, X_test),
    f'SFS ({best_k} features)': (X_train_sfs, X_test_sfs),
    f'T-test ({len(significant_idx)} features)': (X_train_ttest, X_test_ttest),
}

kernels = ['linear', 'rbf', 'poly']

results = []

for fs_name, (X_tr, X_te) in feature_sets.items():
    for kernel in kernels:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel=kernel, class_weight='balanced', random_state=RANDOM_STATE))
        ])

        cv_scores = cross_val_score(pipe, X_tr, y_train, cv=cv, scoring='f1_macro')

        pipe.fit(X_tr, y_train)
        y_pred = pipe.predict(X_te)
        test_f1 = f1_score(y_test, y_pred, average='macro')
        test_bal_acc = balanced_accuracy_score(y_test, y_pred)

        results.append({
            'Feature set': fs_name,
            'Kernel': kernel,
            'CV f1_macro': cv_scores.mean(),
            'CV std': cv_scores.std(),
            'Test f1_macro': test_f1,
            'Test balanced_acc': test_bal_acc
        })

results_df = pd.DataFrame(results).sort_values('CV f1_macro', ascending=False)
print("\n", results_df.round(3).to_string(index=False))

# -----------------------------
# Show detailed report for the best combo
# -----------------------------
best_row = results_df.iloc[0]
print(f"\nBest combo: {best_row['Feature set']} + {best_row['Kernel']} kernel")

X_tr, X_te = feature_sets[best_row['Feature set']]
best_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel=best_row['Kernel'], class_weight='balanced', random_state=RANDOM_STATE))
])
best_pipe.fit(X_tr, y_train)
y_pred = best_pipe.predict(X_te)

print("\nClassification report:")
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

STEP 1: Sequential Feature Selection
k= 2 | features=['coef_7', 'coef_10'] | CV f1_macro=0.508
k= 3 | features=['coef_6', 'coef_7', 'coef_10'] | CV f1_macro=0.509
k= 4 | features=['coef_6', 'coef_7', 'coef_10', 'coef_11'] | CV f1_macro=0.494
k= 5 | features=['coef_4', 'coef_6', 'coef_7', 'coef_10', 'coef_11'] | CV f1_macro=0.500
k= 6 | features=['coef_4', 'coef_6', 'coef_7', 'coef_8', 'coef_10', 'coef_11'] | CV f1_macro=0.520
k= 7 | features=['coef_2', 'coef_4', 'coef_6', 'coef_7', 'coef_8', 'coef_10', 'coef_11'] | CV f1_macro=0.519
k= 8 | features=['coef_1', 'coef_2', 'coef_4', 'coef_6', 'coef_7', 'coef_8', 'coef_10', 'coef_11'] | CV f1_macro=0.506
k= 9 | features=['coef_1', 'coef_2', 'coef_4', 'coef_6', 'coef_7', 'coef_8', 'coef_10', 'coef_11', 'coef_12'] | CV f1_macro=0.462
k=10 | features=['coef_1', 'coef_2', 'coef_3', 'coef_4', 'coef_6', 'coef_7', 'coef_8', 'coef_10', 'coef_11', 'coef_12'] | CV f1_macro=0.461
k=11 | features=['coef_1', 'coef_2', 'coef_3', 'coef_4', 'coef_5', 'coef